# Agrupar 178 vinhos — laboratório avançado
## Machine Learning · Insper

**Para fazer sozinho ou em dupla**, em duas sessões se preciso. Nove etapas, cerca de 2h30.

Você vai comparar **cinco transformações de escala**, **quatro índices internos**,
implementar a **estatística gap** do zero, medir **estabilidade por reamostragem** e
selecionar modelo por **BIC** — para descobrir, na etapa 7, quais desses instrumentos
o levaram à resposta errada, e por quê.

> ### Regra única
> A coluna `df.target` traz a cultivar verdadeira. **Não olhe, não use, não colora
> gráfico com ela antes da etapa 7.** Todas as decisões — escala, k, algoritmo, espaço —
> têm de ser tomadas com os instrumentos que existem quando não há gabarito.

| | |
|---|---|
| Vinhos | 178 |
| Medidas químicas | 13 |
| Razão entre o maior e o menor desvio | 2.530× |
| Critérios de seleção comparados | 5 |


In [1]:
# Tudo o que este laboratório usa já vem instalado no Colab.
import warnings; warnings.filterwarnings("ignore")
from IPython.display import display
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110
print("pronto — siga para a etapa 0")

pronto — siga para a etapa 0


# O terreno, e a regra do laboratório

**Etapa 0 · 10 min**

Este laboratório é **para fazer sozinho ou em dupla**, em duas sessões se preciso. Ele vai bem além do roteiro de aula: você vai comparar **cinco transformações de escala**, **quatro índices internos**, implementar a **estatística gap** do zero, medir **estabilidade por reamostragem** e selecionar modelo por **BIC**.

O conjunto: **178 vinhos italianos** de três cultivares, com **13 medidas químicas**. Vem embutido no scikit-learn — não há download para falhar.

In [2]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_wine

w  = load_wine(as_frame=True)
df = w.frame
F  = w.feature_names

display(df.shape)   # (178, 14)
display(df.isna().sum().sum())   # 0
df[F].std().describe()[["min","max"]].round(3)   # 0.124  e  314.907

(178, 14)

np.int64(0)

min      0.124
max    314.907
dtype: float64

**Verificação:** o maior desvio padrão é **2.530 vezes** o menor. Guarde.

> ##### A coluna `target` fica lacrada até a etapa 7
> 
> Ela traz a cultivar verdadeira. **Não olhe, não use, não colora gráfico com ela** antes da etapa 7. Todas as decisões — escala, k, algoritmo — têm de ser tomadas com os instrumentos que existem quando não há gabarito.
> 
> Um agrupamento com gabarito é um luxo raro. O objetivo aqui é usá-lo para **auditar seus próprios critérios**, não para guiá-los.

# Cinco escalas, e o que os índices dizem sobre elas

**Etapa 1 · 25 min**

Padronizar não é uma decisão binária. Compare cinco tratamentos, medindo os três índices internos usuais.

In [3]:
from sklearn.preprocessing import (StandardScaler, RobustScaler, MinMaxScaler,
                                   PowerTransformer, QuantileTransformer)
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                             davies_bouldin_score)

X = df[F].values
esc = {
    "nenhum":              None,
    "StandardScaler":      StandardScaler(),
    "RobustScaler":        RobustScaler(),
    "MinMaxScaler":        MinMaxScaler(),
    "PowerTransformer":    PowerTransformer(),
    "QuantileTransformer": QuantileTransformer(output_distribution="normal",
                                                 n_quantiles=100, random_state=0),
}
for nome, e in esc.items():
    M = X if e is None else e.fit_transform(X)
    l = KMeans(3, n_init=20, random_state=0).fit_predict(M)
    print(f"{nome:20s} sil {silhouette_score(M,l):.3f}  "
          f"CH {calinski_harabasz_score(M,l):7.1f}  DB {davies_bouldin_score(M,l):.3f}")

nenhum               sil 0.571  CH   561.8  DB 0.534
StandardScaler       sil 0.285  CH    70.9  DB 1.389
RobustScaler         sil 0.264  CH    61.1  DB 1.466
MinMaxScaler         sil 0.301  CH    83.4  DB 1.305
PowerTransformer     sil 0.301  CH    73.1  DB 1.361
QuantileTransformer  sil 0.250  CH    51.6  DB 1.538


| tratamento | silhueta ↑ | Calinski-H. ↑ | Davies-B. ↓ |
|---|---|---|---|
| nenhum | 0,571 | 561,8 | 0,534 |
| StandardScaler | 0,285 | 70,9 | 1,389 |
| RobustScaler | 0,264 | 61,1 | 1,466 |
| MinMaxScaler | 0,301 | 83,4 | 1,305 |
| PowerTransformer | 0,301 | 73,1 | 1,361 |
| QuantileTransformer | 0,250 | 51,6 | 1,538 |

> ##### Os três índices concordam — e os três apontam para a mesma resposta errada
> 
> A silhueta é **máxima** sem escalar. O Calinski-Harabasz é **oito vezes maior** sem escalar. O Davies-Bouldin, que se quer *baixo*, é **2,6 vezes menor** sem escalar. Três instrumentos independentes, unânimes.
> 
> E os três estão errados, como a etapa 7 vai mostrar. O motivo é comum aos três: **todos medem separação geométrica no espaço em que você os aplicou**. Sem escalar, esse espaço é essencialmente o eixo `proline`, e faixas de `proline` são de fato muito bem separadas.
> 
> **A lição:** índices internos servem para comparar *partições do mesmo espaço*. Comparar espaços diferentes com eles é uma inversão — o índice muda porque a régua mudou, não porque a partição melhorou.

**Responda antes de seguir:** se os índices não podem escolher a escala, o que pode? Anote sua resposta; a etapa 3 oferece uma.

Siga com o `StandardScaler`. Ele não é o melhor por nenhum índice, e é a escolha certa pelas razões da aula.

In [4]:
Z = StandardScaler().fit_transform(X)

# Escolher k com quatro instrumentos — e um quinto, feito à mão

**Etapa 2 · 30 min**

#### 2a · Os quatro índices por k

In [5]:
for k in range(2, 9):
    m = KMeans(k, n_init=20, random_state=0).fit(Z)
    print(k, round(m.inertia_,1), round(silhouette_score(Z,m.labels_),4),
             round(calinski_harabasz_score(Z,m.labels_),1),
             round(davies_bouldin_score(Z,m.labels_),4))

2 1659.0 0.2683 69.5 1.4482
3 1277.9 0.2849 70.9 1.3892


4 1180.7 0.2457 55.7 1.7653
5 1108.5 0.2451 47.0 1.736
6 1044.5 0.196 41.8 1.8123
7 987.0 0.1802 38.3 1.804
8 944.6 0.1581 35.2 1.7692


| k | inércia ↓ | silhueta ↑ | CH ↑ | DB ↓ |
|---|---|---|---|---|
| 2 | 1659,0 | 0,2683 | 69,5 | 1,4482 |
| 3 | 1277,9 | 0,2849 | 70,9 | 1,3892 |
| 4 | 1180,7 | 0,2457 | 55,7 | 1,7653 |
| 5 | 1108,5 | 0,2451 | 47,0 | 1,7360 |
| 6 | 1044,5 | 0,1960 | 41,8 | 1,8123 |
| 7 | 987,0 | 0,1802 | 38,3 | 1,8040 |
| 8 | 944,6 | 0,1581 | 35,2 | 1,7692 |

Os três índices comparáveis apontam **k = 3**. Note também que a diferença entre k=2 e k=3 na silhueta é de apenas **0,0166** — uma margem estreita, à qual voltaremos.

#### 2b · A estatística gap, implementada do zero

A ideia de Tibshirani, Walther e Hastie (2001): a inércia sempre cai, então compare a queda observada com **a queda que se obteria em dados sem estrutura**, gerados uniformemente na mesma caixa. O *gap* é a diferença entre os dois, em escala logarítmica.

In [6]:
rng = np.random.default_rng(0)
B   = 20                       # referências uniformes por k
lo, hi = Z.min(axis=0), Z.max(axis=0)

def W(M, k, seed=0):
    return KMeans(k, n_init=10, random_state=seed).fit(M).inertia_

gaps, sks = [], []
for k in range(1, 9):
    logW  = np.log(W(Z, k))
    refs  = [np.log(W(rng.uniform(lo, hi, Z.shape), k)) for _ in range(B)]
    gap   = np.mean(refs) - logW
    sk    = np.std(refs) * np.sqrt(1 + 1/B)      # erro padrão corrigido
    gaps.append(gap); sks.append(sk)
    print(k, round(logW,4), round(gap,4), round(sk,4))

# critério: o menor k tal que gap(k) >= gap(k+1) - s(k+1)
escolha = next(k for k in range(1, 8) if gaps[k-1] >= gaps[k] - sks[k])
print("gap escolhe k =", escolha)   # 3

1 7.7467 0.7903 0.0159


2 7.414 1.012 0.02


3 7.153 1.1988 0.023


4 7.0739 1.2135 0.0234


5 7.0124 1.2341 0.0216


6 6.9513 1.265 0.0223


7 6.9037 1.2693 0.0177


8 6.8507 1.3021 0.0198
gap escolhe k = 3


| k | log W | E[log W*] | gap | s(k) |
|---|---|---|---|---|
| 1 | 7,7467 | 8,5370 | 0,7903 | 0,0159 |
| 2 | 7,4140 | 8,4260 | 1,0120 | 0,0200 |
| 3 | 7,1530 | 8,3518 | 1,1988 | 0,0230 |
| 4 | 7,0739 | 8,2874 | 1,2135 | 0,0234 |
| 5 | 7,0124 | 8,2465 | 1,2341 | 0,0216 |

> ##### Por que o critério não é "onde o gap é máximo"
> 
> O gap continua subindo até k=8 (1,3021). Tomar o máximo daria k=8. O critério correto é **o primeiro k em que o gap já não melhora de forma significativa** — formalmente, o menor k com `gap(k) ≥ gap(k+1) − s(k+1)`.
> 
> Aqui: gap(3) = 1,1988 e gap(4) − s(4) = 1,2135 − 0,0234 = 1,1901. Como 1,1988 ≥ 1,1901, **o critério para em k = 3**.
> 
> Repare no que o gap acrescenta aos outros índices: ele é o único que compara o resultado com **uma hipótese nula explícita** — dados sem estrutura na mesma caixa. É a formalização da "régua do ruído" da aula.

# Estabilidade: o critério que não olha a geometria

**Etapa 3 · 25 min**

Todos os índices até aqui medem *forma*. Há um critério ortogonal a eles: **se os grupos são reais, devem reaparecer quando você reamostra os dados**. Se dependem do sorteio, não são estrutura.

In [7]:
from sklearn.metrics import adjusted_rand_score

rng = np.random.default_rng(0)

def estabilidade(Z, k, B=30, frac=0.8):
    n, labs, idxs = len(Z), [], []
    for b in range(B):
        idx = rng.choice(n, int(frac*n), replace=False)
        labs.append(KMeans(k, n_init=10, random_state=b).fit_predict(Z[idx]))
        idxs.append(idx)
    vals = []
    for a in range(B):
        for c in range(a+1, B):
            com = np.intersect1d(idxs[a], idxs[c])          # pontos nas duas amostras
            if len(com) < 20: continue
            pa = labs[a][[list(idxs[a]).index(x) for x in com]]
            pc = labs[c][[list(idxs[c]).index(x) for x in com]]
            vals.append(adjusted_rand_score(pa, pc))         # concordam entre si?
    return np.mean(vals), np.std(vals)

for k in range(2, 8):
    m, s = estabilidade(Z, k)
    print(k, round(m, 4), round(s, 4))

2 0.7059 0.2755


3 0.9544 0.0382


4 0.8179 0.0994


5 0.7329 0.0987


6 0.6617 0.0995


7 0.5989 0.1076


| k | ARI médio entre pares | desvio |
|---|---|---|
| 2 | 0,7059 | 0,2755 |
| 3 | 0,9544 | 0,0382 |
| 4 | 0,8179 | 0,0994 |
| 5 | 0,7329 | 0,0987 |
| 6 | 0,6617 | 0,0995 |
| 7 | 0,5989 | 0,1076 |

> ##### Este é o instrumento mais decisivo do laboratório
> 
> Em **k = 3**, duas subamostras de 80% concordam com ARI **0,954**, e o desvio é de apenas 0,038 — praticamente sempre a mesma partição.
> 
> Em **k = 2**, a média cai para 0,706 e o **desvio explode para 0,276**. Ou seja: dependendo de quais 80% você sortear, a partição em dois grupos muda radicalmente. Lembre que na etapa 2 a silhueta separava k=2 de k=3 por meros 0,0166. **A estabilidade separa os dois de forma inequívoca**, e na direção certa.
> 
> É um critério com uma vantagem conceitual: **ele não pressupõe nenhuma forma de grupo**. Funciona igual para k-means, GMM ou qualquer outro método, e responde a uma pergunta que nenhum índice geométrico responde — "isto se repete?".

**Responda:** volte à pergunta da etapa 1. Você poderia usar estabilidade para escolher a *escala*? O que precisaria mudar no procedimento?

# Ler a partição: silhueta por ponto e perfil dos centróides

**Etapa 4 · 20 min**

#### 4a · O diagrama de silhueta

In [8]:
from sklearn.metrics import silhouette_samples

km = KMeans(3, n_init=20, random_state=0).fit(Z)
l  = km.labels_
s  = silhouette_samples(Z, l)

for j in range(3):
    print(j, (l==j).sum(), round(s[l==j].mean(),3), round((s[l==j]<0).mean()*100,1))
# 0  65  0.177  10.8
# 1  51  0.351   0.0
# 2  62  0.343   0.0

0 65 0.177 10.8
1 51 0.351 0.0
2 62 0.343 0.0


Um grupo tem metade da coesão dos outros e **7 pontos negativos**. Guarde a hipótese: ele é um grupo mal formado, ou apenas um grupo disperso?

#### 4b · O que caracteriza cada grupo, em unidades originais

Centróides padronizados não se interpretam. Desfaça a transformação:

In [9]:
sc  = StandardScaler().fit(X)
Z   = sc.transform(X)
km  = KMeans(3, n_init=20, random_state=0).fit(Z)
cen = sc.inverse_transform(km.cluster_centers_)      # de volta às unidades reais
pd.DataFrame(cen, columns=F).round(2)

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,12.25,1.90,2.23,20.06,92.74,2.25,2.05,0.36,1.62,2.97,1.06,2.80,510.17
1,13.13,3.31,2.42,21.24,98.67,1.68,0.82,0.45,1.15,7.23,0.69,1.70,619.06
2,13.68,2.00,2.47,17.46,107.97,2.85,3.00,0.29,1.92,5.45,1.07,3.16,1100.23


| grupo | álcool | flavonoides | cor | prolina | OD280 |
|---|---|---|---|---|---|
| 0 | 12,25 | 2,05 | 2,97 | 510 | 2,80 |
| 1 | 13,13 | 0,82 | 7,23 | 619 | 1,70 |
| 2 | 13,68 | 3,00 | 5,45 | 1100 | 3,16 |

Agora os grupos têm descrição: o **1** é o vinho de cor intensa e poucos flavonoides; o **2** é o de prolina alta e mais alcoólico; o **0** é o mais leve em tudo.

#### 4c · Quais variáveis fizeram o trabalho

In [10]:
from scipy.stats import f_oneway
Fs = {c: f_oneway(*[Z[l==j, i] for j in range(3)])[0] for i, c in enumerate(F)}
sorted(Fs.items(), key=lambda x: -x[1])[:5]

[('flavanoids', np.float64(271.58982019939003)),
 ('od280/od315_of_diluted_wines', np.float64(221.52835879582236)),
 ('proline', np.float64(200.3120068360184)),
 ('alcohol', np.float64(113.16607091042137)),
 ('color_intensity', np.float64(111.92310489474306))]

| mais discriminantes | F | menos discriminantes | F |
|---|---|---|---|
| `flavanoids` | 271,6 | `ash` | 14,9 |
| `od280/od315` | 221,5 | `magnesium` | 22,7 |
| `proline` | 200,3 | `alcalinity_of_ash` | 24,5 |

> ##### Cuidado com o que esse F significa
> 
> Ele foi calculado com os **rótulos que o próprio k-means produziu**. Não é um teste de hipótese válido — os grupos foram construídos para maximizar separação, então F alto é esperado por construção. O valor-p aqui **não tem interpretação**.
> 
> O que a lista serve para: **ordenar** as variáveis por contribuição à partição encontrada, como descrição. Nunca como evidência de que a variável "difere entre os grupos" no sentido inferencial.

# Modelo probabilístico: GMM, covariância e BIC

**Etapa 5 · 25 min**

O k-means assume grupos esféricos de tamanho parecido. A mistura de gaussianas relaxa isso: cada componente tem **média e covariância próprias**, e cada ponto recebe uma *probabilidade* de pertencer a cada grupo. O preço é o número de parâmetros — e é aí que entra o BIC.

In [11]:
from sklearn.mixture import GaussianMixture

for tipo in ["full", "tied", "diag", "spherical"]:
    for k in range(2, 7):
        g = GaussianMixture(k, covariance_type=tipo, random_state=0, n_init=5).fit(Z)
        print(tipo, k, round(g.bic(Z), 1), round(g.aic(Z), 1))

full 2 5608.3 4943.4
full 3 5806.3 4807.2
full 4 5670.2 4337.1
full 5 5915.5 4248.3


full 6 6018.8 4017.5
tied 2 5643.3 5267.8
tied 3 5572.6 5152.6


tied 4 5545.3 5080.8
tied 5 5496.3 4987.2
tied 6 5491.8 4938.2
diag 2 5987.9 5819.3
diag 3 5543.4 5288.8
diag 4 5476.5 5136.0
diag 5 5576.8 5150.4


diag 6 5550.6 5038.3
spherical 2 6135.0 6042.7
spherical 3 5708.8 5568.8
spherical 4 5654.9 5467.2


spherical 5 5644.4 5409.0
spherical 6 5647.3 5364.1


| tipo | k = 2 | k = 3 | k = 4 |
|---|---|---|---|
| full | 5608,3 | 5806,3 | 5670,2 |
| tied | 5643,3 | 5572,6 | 5545,3 |
| diag | 5987,9 | 5543,4 | 5476,5 |
| spherical | 6135,0 | 5708,8 | 5654,9 |

> ##### O menor BIC aponta `diag` com k = 4. E erra.
> 
> O BIC mínimo de toda a tabela é **5476,5**, em `covariance_type="diag"` com **k = 4**. Seguindo o critério, essa seria a escolha.
> 
> Na etapa 7 você verá que essa configuração dá ARI **0,683**, enquanto `tied` com k = 3 dá **0,912** — o melhor resultado de todo o laboratório.
> 
> Por quê: o BIC otimiza **verossimilhança penalizada por número de parâmetros**. Com 178 pontos em 13 dimensões, ele prefere modelos mais simples por componente e compensa com mais componentes. **Descrever bem a densidade dos dados e recuperar os grupos que nos interessam são objetivos diferentes.**
> 
> Compare com a etapa 3: a estabilidade apontou k=3 sem ambiguidade. Vale rodar `estabilidade()` com GMM no lugar do k-means e ver o que dá.

#### 5b · A vantagem real do GMM: incerteza por ponto

In [12]:
g = GaussianMixture(3, covariance_type="tied", random_state=0, n_init=5).fit(Z)
p = g.predict_proba(Z)
conf = p.max(axis=1)                       # confiança da atribuição
print((conf < 0.9).sum(), round(conf.min(), 3))
# quantos pontos são ambíguos, e o mais incerto de todos

1 0.58


**Verificação:** apenas **1 ponto** fica abaixo de 0,9 de confiança, e o mais incerto de todos tem 0,58.

> ##### Cruze com a silhueta — e prepare-se para uma surpresa
> 
> Seria natural esperar que os pontos de baixa confiança fossem os de silhueta baixa. **Não são.** A correlação entre as duas medidas é de apenas **0,144**, e dos 15 pontos menos confiantes no GMM, só **5** estão entre os 15 de pior silhueta.
> 
> Faz sentido quando se olha o que cada uma pergunta. A silhueta compara *distâncias médias a conjuntos de pontos*; a confiança do GMM compara *densidades de probabilidade* sob gaussianas ajustadas. Um ponto na fronteira geométrica pode estar em região de alta densidade de uma componente e receber probabilidade quase 1.
> 
> **Duas medidas de incerteza que discordam são mais informativas que uma só** — mas exigem que você saiba o que cada uma está medindo antes de reportá-las.

# Agrupar no espaço reduzido: quando ajuda, quando engana

**Etapa 6 · 20 min**

É prática comum reduzir com PCA antes de agrupar. Teste quantas componentes fazem diferença — e, ao mesmo tempo, onde a métrica deve ser medida.

In [13]:
from sklearn.decomposition import PCA

for nc in [2, 3, 5, 8, 13]:
    A  = PCA(nc, random_state=0).fit_transform(Z)
    la = KMeans(3, n_init=20, random_state=0).fit_predict(A)
    print(nc, round(silhouette_score(A, la), 3),      # medida no espaço reduzido
             round(silhouette_score(Z, la), 3))       # medida no espaço completo

2 0.561 0.283
3 0.453 0.284
5 0.369 0.285
8 0.315 0.285
13 0.285 0.285


| componentes | % da variância | silhueta no PCA | silhueta no 13D |
|---|---|---|---|
| 2 | 55,4% | 0,561 | 0,283 |
| 3 | 66,5% | 0,453 | 0,284 |
| 5 | 80,2% | 0,369 | 0,285 |
| 8 | 92,0% | 0,315 | 0,285 |
| 13 | 100,0% | 0,285 | 0,285 |

> ##### Duas leituras, e a segunda é a que importa
> 
> **A coluna da direita é praticamente constante** — 0,283 a 0,285. Reduzir de 13 para 2 componentes *não mudou a partição* de forma relevante; a informação que separa os grupos está nas primeiras componentes.
> 
> **A coluna do meio quase dobra** conforme se reduz. Não é a partição melhorando: é a régua encurtando. Medida em 2 dimensões, a silhueta ignora as 11 direções em que os grupos se sobrepõem.
> 
> **Regra:** reduza para visualizar e para acelerar, se quiser — mas **meça sempre no espaço em que pretende afirmar o resultado**. Uma silhueta de 0,561 obtida de um gráfico de PCA é um número inflado, e o leitor não tem como perceber.

# Abrir o gabarito e auditar cada decisão

**Etapa 7 · 20 min**

Agora sim. Você tomou quatro decisões — escala, k, algoritmo, espaço. Vamos avaliar as quatro.

In [14]:
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from sklearn.cluster import AgglomerativeClustering, HDBSCAN

y = df.target
particoes = {
    "k-means k=3":  KMeans(3, n_init=20, random_state=0).fit_predict(Z),
    "k-means k=2":  KMeans(2, n_init=20, random_state=0).fit_predict(Z),
    "ward k=3":     AgglomerativeClustering(3, linkage="ward").fit_predict(Z),
    "GMM tied k=3": GaussianMixture(3, covariance_type="tied", random_state=0, n_init=5).fit_predict(Z),
    "GMM diag k=4": GaussianMixture(4, covariance_type="diag", random_state=0, n_init=5).fit_predict(Z),
    "HDBSCAN":      HDBSCAN(min_cluster_size=10).fit_predict(Z),
    "sem padronizar": KMeans(3, n_init=20, random_state=0).fit_predict(X),
}
for nome, lab in particoes.items():
    print(f"{nome:16s} ARI {adjusted_rand_score(y, lab):.3f}  "
          f"AMI {adjusted_mutual_info_score(y, lab):.3f}")

# k-means k=3     ARI 0.897  AMI 0.875
# k-means k=2     ARI 0.389  AMI 0.498
# ward k=3        ARI 0.790  AMI 0.784
# GMM tied k=3    ARI 0.912  AMI 0.895   <- o melhor de todos
# GMM diag k=4    ARI 0.683  AMI 0.731   <- o que o BIC escolheu
# HDBSCAN         ARI 0.260  AMI 0.347
# sem padronizar  ARI 0.371  AMI 0.423

l = particoes["k-means k=3"]
pd.crosstab(l, y)

k-means k=3      ARI 0.897  AMI 0.875
k-means k=2      ARI 0.389  AMI 0.498
ward k=3         ARI 0.790  AMI 0.784
GMM tied k=3     ARI 0.912  AMI 0.895
GMM diag k=4     ARI 0.683  AMI 0.731
HDBSCAN          ARI 0.260  AMI 0.347
sem padronizar   ARI 0.371  AMI 0.423


target,0,1,2
row_0,,,
0,0,65,0
1,0,3,48
2,59,3,0


| decisão | ARI | o que o instrumento dizia |
|---|---|---|
| sem padronizar | 0,371 | sil 0,571 · CH 561,8 · DB 0,534 — **todos favoráveis** |
| StandardScaler, k-means, k=3 | 0,897 | sil 0,285 · estabilidade 0,954 |
| k=2 | 0,389 | sil 0,268 — a 0,0166 de k=3 |
| ward, k=3 | 0,790 | sil 0,277 |
| GMM `tied`, k=3 | 0,912 | BIC 5572,6 — **não era o mínimo** |
| GMM `diag`, k=4 | 0,683 | BIC 5476,5 — **era o mínimo** |
| HDBSCAN | 0,260 | 2 grupos e 75 pontos como ruído |

|  | classe 0 | classe 1 | classe 2 |
|---|---|---|---|
| grupo 0 | 0 | 65 | 0 |
| grupo 1 | 0 | 3 | 48 |
| grupo 2 | 59 | 3 | 0 |

> ##### Três auditorias, e o que cada uma ensina
> 
> **1 · A escala.** Os três índices internos foram unânimes a favor da resposta que dá ARI 0,371 em vez de 0,897. Nenhum instrumento geométrico poderia ter salvado essa decisão — ela se justifica pelo *argumento* (a distância euclidiana exige unidades comparáveis), não pela medição.
> 
> **2 · O k.** A silhueta separava k=2 de k=3 por 0,0166, uma margem que qualquer ruído amostral apagaria. A **estabilidade** separava por 0,25, com desvios de 0,038 contra 0,276. Quando os instrumentos discordam em confiança, use o mais confiante.
> 
> **3 · O grupo 0.** Ele tinha a pior silhueta (0,177) e 10,8% de pontos negativos — e é **100% puro**: 65 vinhos, todos da classe 1. **Silhueta baixa não é grupo errado; é grupo disperso.** Os 6 erros do k-means estão nos *outros* dois grupos, que tinham silhueta alta.

> ##### E o BIC?
> 
> Ele escolheu `diag` com k=4, que dá ARI 0,683, em vez de `tied` com k=3, que dá **0,912** — o melhor resultado do laboratório. O BIC não errou de acordo com seu próprio objetivo: ele encontrou o modelo que melhor descreve a *densidade* dos dados com o menor número de parâmetros.
> 
> Descrever a densidade e recuperar categorias latentes são objetivos distintos. **Todo critério responde exatamente à pergunta que formaliza** — e cabe a você verificar se é a sua pergunta.

# A régua, e o que você poderia ter relatado

**Etapa 8 · 15 min**

In [15]:
rng = np.random.default_rng(0)
Z_emb = Z.copy()
for j in range(Z_emb.shape[1]):
    rng.shuffle(Z_emb[:, j])       # destrói a relação entre colunas, preserva cada marginal

for k in [2, 3, 4]:
    lb = KMeans(k, n_init=20, random_state=0).fit_predict(Z_emb)
    print(k, round(silhouette_score(Z_emb, lb), 3),
             round(calinski_harabasz_score(Z_emb, lb), 1))
# 2  0.070   ...
# 3  0.068   13.3
# 4  0.072   ...

2 0.07 14.5
3 0.068 13.3
4 0.072 12.5


|  | dados reais | embaralhados | razão |
|---|---|---|---|
| silhueta (k=3) | 0,285 | 0,068 | **4,2×** |
| Calinski-Harabasz (k=3) | 70,9 | 13,3 | **5,3×** |

Uma silhueta de 0,285 parece baixa em termos absolutos. **Sem a régua você não teria como julgar** — e poderia ter descartado um agrupamento que acerta 172 de 178.

> ##### O parágrafo que você escreveria no relatório
> 
> "Padronizamos as 13 variáveis com `StandardScaler`, dado que as escalas diferem por até 2.530×. Particionamos com k-means; escolhemos **k = 3** por três critérios convergentes — silhueta (0,285, máxima em k=3), estatística gap (k=3 pelo critério de Tibshirani) e **estabilidade por reamostragem** (ARI 0,954 ± 0,038 entre subamostras de 80%, contra 0,706 ± 0,276 em k=2). A silhueta obtida compara-se a 0,068 sobre os mesmos dados com colunas permutadas. Um dos três grupos apresenta coesão menor (silhueta média 0,177; 10,8% de pontos com silhueta negativa) e deve ser interpretado com cautela."
> 
> Repare no que esse parágrafo faz: declara o pré-processamento e o motivo, o k e **os três critérios**, a linha de base, e a fragilidade conhecida. Nenhuma frase afirma ter "descoberto" três grupos.

#### Três desafios

**1**Estabilidade para escolher a escala20 min

Na etapa 1 os índices internos falharam em comparar espaços diferentes. A estabilidade tem o mesmo problema? Rode `estabilidade(Z, 3)` para cada uma das cinco transformações e compare.

<details>
<summary><b>Pense antes de abrir</b></summary>

A estabilidade **também** é calculada dentro de um espaço, então a comparação entre espaços continua discutível — mas há uma diferença: ela mede *reprodutibilidade*, não separação, e não é inflada por uma variável de escala grande dominar a distância.

Faça o teste e discuta. Se a estabilidade também favorecer o dado bruto, você terá aprendido que **nenhum critério interno resolve a questão da escala** — ela é uma decisão de modelagem, justificada por argumento sobre as unidades, não por medição.

</details>

**2**Consenso entre execuções25 min

Construa uma **matriz de coassociação**: rode k-means 100 vezes com sementes diferentes e conte, para cada par de vinhos, em quantas execuções eles caíram juntos. Divida por 100. Depois agrupe essa matriz (por exemplo com `AgglomerativeClustering(3, metric="precomputed", linkage="average")` sobre `1 - coassociação`).

Compare o ARI do consenso com o do k-means simples. E olhe os pares com coassociação próxima de 0,5: são exatamente os pontos que a etapa 4 marcou com silhueta negativa?

**3**Quantas variáveis bastam?20 min

Pela ANOVA da etapa 4, `flavanoids`, `od280/od315` e `proline` lideram. Agrupe usando **apenas essas três** e compare o ARI com o das 13.

Depois faça o oposto: agrupe com as três *menos* discriminantes. A diferença entre os dois resultados é o argumento a favor de seleção de atributos antes de agrupar — e o alerta de que a seleção foi feita **usando os próprios grupos**, o que é circular. Como você tornaria o procedimento honesto?

> ##### As cinco lições deste laboratório
> 
> 1. **Índices internos comparam partições, não espaços.** Os três foram unânimes a favor de não padronizar, e os três estavam errados.
> 2. **A estabilidade é o critério mais decisivo aqui** — separou k=2 de k=3 por 0,25 onde a silhueta separava por 0,017.
> 3. **O gap statistic formaliza a régua do ruído**, comparando com uma hipótese nula explícita em vez de um limiar de gosto.
> 4. **Silhueta baixa não é grupo errado.** O grupo de pior silhueta era 100% puro.
> 5. **Todo critério responde à pergunta que formaliza.** O BIC descreve densidade; não prometeu recuperar cultivares, e não recuperou.